# affect_aif Demo Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/har5h1l/affect_aif/blob/main/notebooks/demo.ipynb)

Run small, Colab-compatible trust-task experiments and inspect the generated outputs. This notebook is intentionally demo-scale: it exercises the same public runner and analysis path as the paper workflow, but with fewer seeds and shorter episodes.

Default behavior writes scratch outputs under `outputs/notebook_demo/`, not canonical `results/`, so your clean result packet remains shareable.


## 1. Bootstrap Runtime

This cell works in two modes:

- **Google Colab:** clones the repo branch and moves into it.
- **Local Jupyter:** assumes you launched the notebook from the repo root.

Use a GPU runtime if available, but the demo also runs on CPU.


In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap

IN_COLAB = Path("/content").exists()
REPO_URL = os.environ.get("AFFECT_AIF_REPO_URL", "https://github.com/har5h1l/affect_aif.git")
BRANCH = os.environ.get("AFFECT_AIF_BRANCH", "main")

if IN_COLAB:
    os.chdir("/content")
    if not Path("affect_aif").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "affect_aif"], check=True)
    os.chdir("/content/affect_aif")

ROOT = Path.cwd()
if not (ROOT / "scripts" / "experiment" / "run.py").exists():
    raise RuntimeError("Run this notebook from the affect_aif repo root, or use the Colab bootstrap clone.")

print("Repo root:", ROOT)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


## 2. Install Dependencies

Colab runtimes are fresh virtual machines, so install the repo into the runtime. Local users can set `INSTALL_DEPS = False` if the environment is already prepared.


In [ ]:
INSTALL_DEPS = True

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
    print("Installed affect_aif in editable mode for this runtime.")
else:
    print("Skipped dependency install.")


## 3. Check Runtime Device

This reports whether JAX sees CPU/GPU devices. The trust-task demo does not require a GPU, but the check makes the runtime transparent.


In [ ]:
try:
    import jax
    print("JAX devices:", jax.devices())
except Exception as exc:
    print("JAX device check unavailable:", exc)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU visible. This notebook also runs on CPU; GPU is optional for these trust-task sweeps.")


## 4. Run Demo Experiments

These configs run actual experiments through `scripts/experiment/run.py`:

- model-fitness demo
- betrayal-adaptation demo
- alpha-sweep demo

Set `RUN_DEMO = False` to skip execution and inspect previously generated outputs.


In [ ]:
RUN_DEMO = True
RESET_OUTPUTS = False
OUTPUT_ROOT = Path("outputs")
DEMO_BATCH = "notebook_demo"
DEMO_OUT = OUTPUT_ROOT / DEMO_BATCH
DEMO_CONFIGS = [
    "configs/demo/model_fitness.toml",
    "configs/demo/betrayal_adaptation.toml",
    "configs/demo/alpha_sweep.toml",
]

for config in DEMO_CONFIGS:
    if not Path(config).exists():
        raise FileNotFoundError(config)

if RESET_OUTPUTS and DEMO_OUT.exists():
    shutil.rmtree(DEMO_OUT)

cmd = [sys.executable, "scripts/experiment/run.py"]
for config in DEMO_CONFIGS:
    cmd.extend(["--config", config])
cmd.extend(["--output-dir", str(OUTPUT_ROOT), "--batch-name", DEMO_BATCH, "--workers", "1"])

print("Command:", " ".join(cmd))
if RUN_DEMO:
    subprocess.run(cmd, check=True)
else:
    print("Skipped run. Set RUN_DEMO = True to execute the demo.")


## 5. Run Post-Hoc Analysis

The runner writes per-round `results.csv` files. This cell runs the general analysis CLI for each generated result file and stores summaries beside the scratch outputs.


In [ ]:
RUN_ANALYSIS = True
result_paths = sorted(DEMO_OUT.glob("**/results.csv"))
if not result_paths:
    raise RuntimeError("No demo results found. Run the experiment cell first.")

analysis_dirs = []
for result_path in result_paths:
    analysis_dir = result_path.parent / "notebook_analysis"
    analysis_dirs.append(analysis_dir)
    cmd = [
        sys.executable,
        "scripts/analysis/analyze.py",
        "--results",
        str(result_path),
        "--output-dir",
        str(analysis_dir),
    ]
    if RUN_ANALYSIS:
        subprocess.run(cmd, check=True)

print("Analyzed result files:")
for path in result_paths:
    print("-", path)


## 6. Load Demo Outputs

The inline analysis below combines generated trajectories across demo configs. These summaries are for demonstration only; paper claims use the full `configs/paper/` runs.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

frames = []
for path in result_paths:
    frame = pd.read_csv(path, low_memory=False)
    frame["source_file"] = str(path)
    frames.append(frame)

data = pd.concat(frames, ignore_index=True)
print("Rows:", len(data))
print("Experiments:", sorted(data["experiment_id"].dropna().unique()))
print("Variants:", sorted(data["variant_id"].dropna().unique()))

def summarize(frame):
    aggregations = {"payoff": ["mean", "sum"]}
    if "q_pi_entropy" in frame.columns:
        aggregations["q_pi_entropy"] = ["mean"]
    return frame.groupby(["experiment_id", "variant_id"], as_index=False).agg(aggregations)

demo_summary = summarize(data)
demo_summary


## 7. Model-Fitness Demo Readout

This readout checks whether model-fitness variants differ in payoff and policy entropy at demo scale. Treat it as an execution sanity check, not manuscript evidence.


In [ ]:
model_fitness = data[data["experiment_id"] == "model_fitness_demo"].copy()
if model_fitness.empty:
    print("No model_fitness_demo rows found.")
else:
    mf = model_fitness.groupby("variant_id", as_index=False).agg(
        mean_payoff=("payoff", "mean"),
        total_payoff=("payoff", "sum"),
        mean_entropy=("q_pi_entropy", "mean") if "q_pi_entropy" in model_fitness.columns else ("payoff", "mean"),
    )
    display(mf)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].bar(mf["variant_id"], mf["mean_payoff"])
    axes[0].set(title="Mean payoff", xlabel="Variant", ylabel="Payoff")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].bar(mf["variant_id"], mf["mean_entropy"])
    axes[1].set(title="Mean policy entropy", xlabel="Variant", ylabel="Entropy")
    axes[1].tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()
    low_entropy = mf.sort_values("mean_entropy").iloc[0]
    high_payoff = mf.sort_values("mean_payoff", ascending=False).iloc[0]
    print(f"Demo interpretation: lowest entropy = {low_entropy.variant_id}; highest mean payoff = {high_payoff.variant_id}.")


## 8. Betrayal-Adaptation Demo Readout

The scheduled stance switch occurs at round 16. The plot shows how policy entropy evolves before and after that switch.


In [ ]:
betrayal = data[data["experiment_id"] == "betrayal_adaptation_demo"].copy()
if betrayal.empty or "round" not in betrayal.columns:
    print("No betrayal_adaptation_demo rows found.")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    metric = "q_pi_entropy" if "q_pi_entropy" in betrayal.columns else "payoff"
    for variant, frame in betrayal.groupby("variant_id"):
        by_round = frame.groupby("round", as_index=False)[metric].mean()
        ax.plot(by_round["round"], by_round[metric], label=variant)
    ax.axvline(16, color="black", linestyle="--", linewidth=1, label="stance switch")
    ax.set(title=f"Betrayal demo: {metric} by round", xlabel="Round", ylabel=metric)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

    window = betrayal.assign(phase=lambda d: pd.cut(d["round"], bins=[-1, 15, 50], labels=["pre", "post"]))
    phase_summary = window.groupby(["variant_id", "phase"], observed=True)[metric].mean().reset_index()
    display(phase_summary)
    print("Demo interpretation: compare pre/post rows to see which variants sharpen or soften policy deployment after betrayal.")


## 9. Alpha-Sweep Demo Readout

Alpha controls affective precision gain. The demo checks whether changing gain visibly changes entropy or payoff in a shortened setting.


In [ ]:
alpha = data[data["experiment_id"] == "alpha_sweep_demo"].copy()
if alpha.empty:
    print("No alpha_sweep_demo rows found.")
else:
    alpha_summary = alpha.groupby("variant_id", as_index=False).agg(
        mean_payoff=("payoff", "mean"),
        total_payoff=("payoff", "sum"),
        mean_entropy=("q_pi_entropy", "mean") if "q_pi_entropy" in alpha.columns else ("payoff", "mean"),
    )
    display(alpha_summary)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].bar(alpha_summary["variant_id"], alpha_summary["mean_payoff"])
    axes[0].set(title="Alpha demo: mean payoff", xlabel="Variant", ylabel="Payoff")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].bar(alpha_summary["variant_id"], alpha_summary["mean_entropy"])
    axes[1].set(title="Alpha demo: mean entropy", xlabel="Variant", ylabel="Entropy")
    axes[1].tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()
    print("Demo interpretation: stronger gain should be read as a reactivity manipulation; the full paper sweep is in configs/paper/alpha_sweep.toml.")


## 10. Optional Drive Export

In Colab, uncomment this cell to copy demo outputs to Google Drive. For paper results, use `notebooks/reproduce.ipynb` and export the canonical `results/` folder.


In [ ]:
EXPORT_TO_DRIVE = False
if EXPORT_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive export is only available in Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dest = Path("/content/drive/MyDrive/affect_aif_notebook_demo")
    drive_dest.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-a", str(DEMO_OUT) + "/", str(drive_dest) + "/"], check=True)
    print("Copied demo outputs to", drive_dest)
else:
    print("Drive export skipped.")
